In [1]:
# Clone your repo
!git clone https://github.com/RiteshGupta-02/chestnut.git
%cd chestnut

Cloning into 'chestnut'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 116 (delta 27), reused 37 (delta 15), pack-reused 66 (from 1)
Receiving objects: 100% (116/116), 309.49 MiB | 44.33 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/kaggle/working/chestnut


In [2]:
!pip install -q opencv-python-headless
# opencv-python-headless = server version of OpenCV, no display needed
# -q = quiet, less output noise

In [3]:
import torch
import torchvision
import sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")   # must print True
print(f"GPU name : {torch.cuda.get_device_name(0)}")    # Tesla T4

PyTorch  : 2.10.0+cu128
CUDA available: True
GPU name : Tesla T4


In [4]:
import sys
sys.path.insert(0, "/kaggle/working/chexnet/src")  # so Python finds your modules

# Kaggle paths
DATA_DIR       = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/"
CHECKPOINT_DIR = "/kaggle/working/chestnut/src/checkpoints/"
LOG_DIR        = "/kaggle/working/chestnut/logs/"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("Paths set up ✓")

Paths set up ✓


In [5]:
from src.chestxray_dataset import get_dataloaders
from pathlib import Path

# Small test — batch_size=4, just verify it runs
train_loader, val_loader, test_loader, pos_weights = get_dataloaders(
    data_dir=Path(DATA_DIR),
    batch_size=4,
    num_workers=2,
)

images, labels = next(iter(train_loader))
print(f"Image shape : {images.shape}")    # (4, 3, 224, 224)
print(f"Label shape : {labels.shape}")    # (4, 14)
print(f"pos_weights : {pos_weights.shape}")  # (14,)
print("Data loading ✓")

[Dataset] Loaded 112,120 rows from /kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv
[Dataset] Class distribution:
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
dtype: int64

[Split] Train: 77,988  |  Val: 8,536  |  Test: 25,596
[Weights] Positive class weights:
  Atelectasis           : 9.5
  Cardiomegaly          : 49.3
  Effusion              : 8.9
  Infiltration          : 5.2
  Mass                  : 20.4
  Nodule                : 17.4
  Pneumonia             : 50.0
  Pneumothorax          : 31.6
  Consolidation         : 29.2
  Edema                 : 50.0
  Emphysema             : 50.0
  Fibrosis              :

In [6]:
!git pull

Already up to date.


In [7]:
from model import get_model
from train import get_optimizer, get_criterion, train_one_epoch, evaluate, save_checkpoint
from pathlib import Path
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Full training settings
train_loader, val_loader, test_loader, pos_weights = get_dataloaders(
    data_dir=Path(DATA_DIR),
    batch_size=32,       # T4 can handle 32 comfortably
    num_workers=4,       # parallel data loading
)

model     = get_model(num_classes=14, device=device)
optimizer = get_optimizer(model)
criterion = get_criterion(pos_weights, device)

# Learning rate scheduler — reduces lr when val loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.1
)

best_val_loss   = float('inf')
patience_count  = 0
EARLY_STOP      = 5
EPOCHS          = 10


for epoch in range(1, EPOCHS + 1):
    try:
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        val_loss   = evaluate(model, val_loader, criterion, device, epoch)
        scheduler.step(val_loss)
    
        print(f"Epoch {epoch}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
    
        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            save_checkpoint(model, epoch, val_loss, CHECKPOINT_DIR)
            print(f"  ✓ New best saved (val_loss: {val_loss:.4f})")
        else:
            patience_count += 1
            print(f"  No improvement ({patience_count}/{EARLY_STOP})")
            if patience_count >= EARLY_STOP:
                print("Early stopping triggered.")
                break
    except Exception as e:
        print(e)
        continue

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

Training on: cuda
[Dataset] Loaded 112,120 rows from /kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv
[Dataset] Class distribution:
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
dtype: int64

[Split] Train: 77,988  |  Val: 8,536  |  Test: 25,596
[Weights] Positive class weights:
  Atelectasis           : 9.5
  Cardiomegaly          : 49.3
  Effusion              : 8.9
  Infiltration          : 5.2
  Mass                  : 20.4
  Nodule                : 17.4
  Pneumonia             : 50.0
  Pneumothorax          : 31.6
  Consolidation         : 29.2
  Edema                 : 50.0
  Emphysema             : 50.0
  Fibro

100%|██████████| 30.8M/30.8M [00:00<00:00, 209MB/s]


Epoch 1/10 | Train: 0.9825 | Val: 0.9037
  Checkpoint saved /kaggle/working/chestnut/src/checkpoints//checkpoint_epoch1.tar
  ✓ New best saved (val_loss: 0.9037)
Epoch 2/10 | Train: 0.9004 | Val: 0.8891
  Checkpoint saved /kaggle/working/chestnut/src/checkpoints//checkpoint_epoch2.tar
  ✓ New best saved (val_loss: 0.8891)
Epoch 3/10 | Train: 0.8689 | Val: 0.8631
  Checkpoint saved /kaggle/working/chestnut/src/checkpoints//checkpoint_epoch3.tar
  ✓ New best saved (val_loss: 0.8631)
Epoch 4/10 | Train: 0.8429 | Val: 0.8583
  Checkpoint saved /kaggle/working/chestnut/src/checkpoints//checkpoint_epoch4.tar
  ✓ New best saved (val_loss: 0.8583)
Epoch 5/10 | Train: 0.8209 | Val: 0.8679
  No improvement (1/5)
Epoch 6/10 | Train: 0.8050 | Val: 0.8623
  No improvement (2/5)
Epoch 7/10 | Train: 0.7844 | Val: 0.8720
  No improvement (3/5)
Epoch 8/10 | Train: 0.7216 | Val: 0.8511
  Checkpoint saved /kaggle/working/chestnut/src/checkpoints//checkpoint_epoch8.tar
  ✓ New best saved (val_loss: 0.8511

In [8]:
with open("/kaggle/working/chestnut/src/chestxray_dataset.py","r") as f:
    print(f.readlines(),sep="\n")

['"""\n', 'NIH ChestX-ray14 Dataset — PyTorch Dataset class\n', 'Multi-label classification with augmentation for CheXNet replication\n', '\n', 'Expected directory structure:\n', '    data/\n', '    ├── images/                   # all unzipped .png X-ray images\n', '    ├── Data_Entry_2017.csv       # main labels file\n', '    ├── train_val_list.txt        # official train split\n', '    └── test_list.txt             # official test split\n', '\n', 'Usage:\n', '    from chestxray_dataset import ChestXray14Dataset, get_transforms, get_dataloaders\n', "    train_loader, val_loader, test_loader = get_dataloaders('data/')\n", '"""\n', '\n', 'import os\n', 'from pathlib import Path\n', 'import numpy as np\n', 'import pandas as pd\n', 'from PIL import Image\n', 'from sklearn.model_selection import train_test_split\n', '\n', 'import torch\n', 'from torch.utils.data import Dataset, DataLoader\n', 'import torchvision.transforms as T\n', '\n', 'os.chdir(Path(__file__).resolve().parent)\n', '\n',